# di.simbasket — Factor Model Estimation

Estimates Fama-French 3-factor model parameters for a basket of 5 stocks.

**Outputs:**
- `factors.csv` — 3×3 factor correlation matrix
- `factorvols.csv` — annualized vol per factor
- A summary table of betas and vols to populate `basket.csv` manually

**Factors:** market (Mkt-RF), value (HML), profitability (RMW)  
**Stocks:** NVDA, AMZN, JPM, JNJ, XOM  
**Estimation window:** 2023-01-01 to 2024-12-31

In [1]:
import io, zipfile, os
import numpy as np
import pandas as pd
import requests
import statsmodels.api as sm
import yfinance as yf

STOCKS  = ['NVDA', 'AMZN', 'JPM', 'JNJ', 'XOM']
FACTORS = ['market', 'value', 'profitability']
FF_COLS = ['Mkt-RF', 'HML', 'RMW']
START   = '2023-01-01'
END     = '2024-12-31'

OUT = os.path.dirname(os.path.abspath('estimate_basket.ipynb'))  # simbasket/notebooks/

## 1. Download stock prices

In [2]:
raw = yf.download(STOCKS, start=START, end=END, auto_adjust=True, progress=False)
prices  = raw['Close']
returns = prices.pct_change().dropna()
stock_vols    = returns.std() * np.sqrt(252)
latest_prices = prices.iloc[-1]

print(f'{len(returns)} trading days')
pd.DataFrame({
    'latest_price': latest_prices.round(2),
    'ann_vol':      stock_vols.round(4),
})

500 trading days


,latest_price,ann_vol
Ticker,,
AMZN,221.30,0.3064
JNJ,138.35,0.1584
JPM,233.31,0.2219
NVDA,137.44,0.5050
XOM,101.34,0.2220


## 2. Download Fama-French factors

In [3]:
FF_URL = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip'
resp = requests.get(FF_URL, timeout=30)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    fname = [f for f in z.namelist() if f.lower().endswith('.csv')][0]
    with z.open(fname) as f:
        raw_ff = f.read().decode('utf-8')

lines = raw_ff.splitlines()
start_idx = next(i for i, l in enumerate(lines) if l.strip().startswith('19'))

# Strip footer — keep only lines where first field is a valid 8-digit date
data_lines = [l for l in lines[start_idx:] if l.strip()[:8].isdigit()]

ff = pd.read_csv(
    io.StringIO('\n'.join(data_lines)),
    header=None, names=['date', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF'], index_col=0,
)
ff.index = pd.to_datetime(ff.index, format='%Y%m%d')
ff = ff.apply(pd.to_numeric, errors='coerce') / 100
ff = ff.loc[START:END, FF_COLS]

print(f'{len(ff)} factor days loaded')
ff.describe().round(4)

502 factor days loaded


,Mkt-RF,HML,RMW
count,502.0000,502.0000,502.0000
mean,0.0008,-0.0004,0.0002
std,0.0085,0.0075,0.0051
min,-0.0317,-0.0360,-0.0156
25%,-0.0038,-0.0056,-0.0031
50%,0.0005,-0.0005,0.0004
75%,0.0062,0.0046,0.0036
max,0.0291,0.0350,0.0179


## 3. Estimate betas (OLS)

In [4]:
data = returns.join(ff, how='inner')
stock_rets  = data[STOCKS]
factor_rets = data[FF_COLS]
print(f'{len(data)} common trading days\n')

betas = {}
rows  = []
for sym in STOCKS:
    model = sm.OLS(stock_rets[sym], sm.add_constant(factor_rets)).fit()
    betas[sym] = model.params[FF_COLS].values
    rows.append({
        'sym':          sym,
        'beta_market':  round(betas[sym][0], 4),
        'beta_value':   round(betas[sym][1], 4),
        'beta_profit':  round(betas[sym][2], 4),
        'R²':           round(model.rsquared, 3),
    })

pd.DataFrame(rows).set_index('sym')

500 common trading days



,beta_market,beta_value,beta_profit,R²
sym,,,,
NVDA,2.2485,-1.4755,1.2187,0.468
AMZN,1.3731,-0.6920,-0.0751,0.490
JPM,0.9213,0.9107,-0.0932,0.488
JNJ,0.1660,0.3224,-0.0307,0.070
XOM,0.5341,0.8387,0.3017,0.274


## 4. Factor correlation matrix and vols

In [5]:
factor_corr     = factor_rets.corr().values
factor_vols_ann = factor_rets.std().values * np.sqrt(252)

print('Factor vols (annualized):')
for name, vol in zip(FACTORS, factor_vols_ann):
    print(f'  {name}: {vol:.4f}')

print('\nFactor correlation matrix:')
pd.DataFrame(factor_corr.round(4), index=FACTORS, columns=FACTORS)

Factor vols (annualized):
  market: 0.1345
  value: 0.1196
  profitability: 0.0814

Factor correlation matrix:


,market,value,profitability
market,1.0000,-0.1343,-0.3527
value,-0.1343,1.0000,0.1749
profitability,-0.3527,0.1749,1.0000


## 5. Idiosyncratic vol check

Verifies that `idiovol² = vol² - beta' * F * beta > 0` for each stock.  
If negative, the betas explain more variance than the stock actually has — betas need shrinking.

In [6]:
D = np.diag(factor_vols_ann)
F = D @ factor_corr @ D   # factor covariance matrix

rows = []
for sym in STOCKS:
    B             = betas[sym]
    total_var     = stock_vols[sym] ** 2
    explained_var = B @ F @ B
    idio_var      = total_var - explained_var
    rows.append({
        'sym':       sym,
        'total_vol': round(float(stock_vols[sym]), 4),
        'explained': round(float(np.sqrt(max(explained_var, 0))), 4),
        'idiovol':   round(float(np.sqrt(max(idio_var, 0))), 4),
        'ok':        '✓' if idio_var >= 0 else '⚠ shrink betas',
    })

pd.DataFrame(rows).set_index('sym')

,total_vol,explained,idiovol,ok
sym,,,,
NVDA,0.5050,0.3456,0.3682,✓
AMZN,0.3064,0.2146,0.2187,✓
JPM,0.2219,0.1550,0.1588,✓
JNJ,0.1584,0.0420,0.1527,✓
XOM,0.2220,0.1162,0.1891,✓


## 6. Summary — values to copy into basket.csv

Copy `startprice`, `vol`, `beta_market`, `beta_value`, `beta_profitability` into `basket.csv`.

In [7]:
summary = pd.DataFrame([{
    'sym':                 sym,
    'startprice':          round(float(latest_prices[sym]), 2),
    'vol':                 round(float(stock_vols[sym]), 4),
    'beta_market':         round(float(betas[sym][0]), 4),
    'beta_value':          round(float(betas[sym][1]), 4),
    'beta_profitability':  round(float(betas[sym][2]), 4),
} for sym in STOCKS]).set_index('sym')

summary

,startprice,vol,beta_market,beta_value,beta_profitability
sym,,,,,
NVDA,137.44,0.5050,2.2485,-1.4755,1.2187
AMZN,221.30,0.3064,1.3731,-0.6920,-0.0751
JPM,233.31,0.2219,0.9213,0.9107,-0.0932
JNJ,138.35,0.1584,0.1660,0.3224,-0.0307
XOM,101.34,0.2220,0.5341,0.8387,0.3017


## 7. Write factors.csv and factorvols.csv

In [8]:
# factors.csv
factors_df = pd.DataFrame(factor_corr.round(6), index=FACTORS, columns=FACTORS)
factors_df.index.name = 'factor'
factors_df.to_csv(os.path.join(OUT, 'factors.csv'))

# factorvols.csv
factorvols_df = pd.DataFrame({'factor': FACTORS, 'vol': factor_vols_ann.round(6)})
factorvols_df.to_csv(os.path.join(OUT, 'factorvols.csv'), index=False)

print(f'Written to {OUT}:')
print('  factors.csv')
print('  factorvols.csv')
print('\nNext: copy the summary table above into basket.csv')

Written to /home/philippe/kdbx-modules/di/simbasket/notebooks:
  factors.csv
  factorvols.csv

Next: copy the summary table above into basket.csv
